# Envoltória resistente de seções de concreto armado — passo a passo

Este notebook mostra, etapa por etapa, como o software calcula a **envoltória resistente N × M** (diagrama de interação), o **momento resistente MRd**, a **armadura necessária**, a **flexão oblíqua Mx × My** e as **envoltórias mínimas** de 1ª e 2ª ordem.

As funções têm os mesmos nomes das usadas nos motores do software (`motor/*.py`), para facilitar a comparação.

**Referência principal:** ABNT NBR 6118:2023 — *Projeto de estruturas de concreto*. Os números de item citados seguem a numeração da norma; recomenda-se conferir na sua cópia.

**Exemplo:** fuste do tubulão sem base, solo I — D = 60 cm, 13 Ø 16 mm, C25, CA-50.

**Convenções:** compressão positiva; tensões em kgf/cm², forças em kgf, comprimentos em cm e momentos em kgf·cm nos cálculos internos (os resultados são mostrados em tf e tf·m).

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25,
                     'axes.spines.top': False, 'axes.spines.right': False})
AZUL, VERDE, LARANJA, CINZA = '#2a5c9a', '#1f8a5b', '#d4691e', '#4a4e53'

## 1. Materiais e resistências de cálculo

**Concreto** — resistência de cálculo (NBR 6118, 12.3.3):

$$f_{cd} = \frac{f_{ck}}{\gamma_c}$$

**Aço** — resistência de cálculo (12.3.3) e módulo de elasticidade $E_s = 210$ GPa (8.3.5):

$$f_{yd} = \frac{f_{yk}}{\gamma_s} \qquad \varepsilon_{yd} = \frac{f_{yd}}{E_s}$$

> No projeto o tubulão usa γc = 2,2 (valor adotado na planilha de referência para concreto de tubulão, sem controle de execução como em peças convencionais). Para as demais peças usa-se γc = 1,4 (12.4.1).

Unidades: o software trabalha em kgf/cm² usando 1 MPa ≈ 10 kgf/cm².

In [ ]:
# Dados do exemplo (tubulão sem base, solo I)
fck = 25.0          # MPa
fyk = 500.0         # MPa (CA-50)
gamma_c = 2.2       # tubulão (planilha de referência)
gamma_s = 1.15
D = 60.0            # cm — diâmetro do fuste
cobrimento = 5.0    # cm
phi_estribo = 6.3   # mm
phi = 16.0          # mm — bitola N1
n_barras = 13

fcd = fck / gamma_c * 10            # kgf/cm²
fyd = fyk / gamma_s * 10            # kgf/cm²
ES_KGF_CM2 = 2100000.0              # Es = 210 GPa
eps_yd = fyd / ES_KGF_CM2
print(f'fcd = {fcd:.1f} kgf/cm²   fyd = {fyd:.1f} kgf/cm²   εyd = {eps_yd*1000:.3f} ‰')

## 2. Diagrama tensão-deformação do concreto — parábola-retângulo

**NBR 6118, 8.2.10.1** — para análise no estado-limite último:

$$\sigma_c = \alpha_c\, f_{cd}\left[1-\left(1-\frac{\varepsilon_c}{\varepsilon_{c2}}\right)^n\right] \quad \text{para } 0 \le \varepsilon_c \le \varepsilon_{c2}; \qquad \sigma_c = \alpha_c f_{cd} \text{ para } \varepsilon_{c2} < \varepsilon_c \le \varepsilon_{cu}$$

| Parâmetro | fck ≤ 50 MPa | 50 < fck ≤ 90 MPa |
|---|---|---|
| εc2 | 2,0 ‰ | 2,0 ‰ + 0,085 ‰ (fck − 50)^0,53 |
| εcu | 3,5 ‰ | 2,6 ‰ + 35 ‰ [(90 − fck)/100]^4 |
| n | 2 | 1,4 + 23,4 [(90 − fck)/100]^4 |

O fator **αc = 0,85** (para fck ≤ 50 MPa) considera o efeito Rüsch — perda de resistência sob carga de longa duração — e é definido em **17.2.2**: αc = 0,85·[1 − (fck − 50)/200] acima de C50.

O concreto **não resiste à tração** no ELU (17.2.2, hipótese c).

In [ ]:
def parametros_concreto(fck_mpa):
    # NBR 6118:2023, 8.2.10.1 — εc2, εcu e expoente n (fck em MPa).
    if fck_mpa <= 50:
        return 0.002, 0.0035, 2.0
    k = (90 - fck_mpa) / 100
    return 0.002 + 0.000085 * (fck_mpa - 50) ** 0.53, 0.0026 + 0.035 * k ** 4, 1.4 + 23.4 * k ** 4


def alfa_c(fck_mpa):
    # NBR 6118:2023, 17.2.2 — fator do bloco de tensões.
    return 0.85 if fck_mpa <= 50 else 0.85 * (1 - (fck_mpa - 50) / 200)


def tensao_concreto(eps, sigma_cd, ec2, nexp):
    # Parábola-retângulo; tração (eps < 0) não resiste.
    ec = np.clip(eps, 0.0, ec2)
    return np.where(eps > 0, sigma_cd * (1 - (1 - ec / ec2) ** nexp), 0.0)


ec2, ecu, nexp = parametros_concreto(fck)
sigma_cd = alfa_c(fck) * fcd
print(f'εc2 = {ec2*1000:.2f} ‰   εcu = {ecu*1000:.2f} ‰   n = {nexp}   αc·fcd = {sigma_cd:.1f} kgf/cm²')

e = np.linspace(-0.001, ecu, 300)
plt.figure(figsize=(6, 3.5))
plt.plot(e * 1000, tensao_concreto(e, sigma_cd, ec2, nexp), color=AZUL, lw=2)
plt.axvline(ec2 * 1000, ls='--', color=CINZA, lw=1); plt.axvline(ecu * 1000, ls='--', color=CINZA, lw=1)
plt.text(ec2 * 1000, 5, ' εc2', color=CINZA); plt.text(ecu * 1000, 5, ' εcu', color=CINZA)
plt.xlabel('εc (‰)'); plt.ylabel('σc (kgf/cm²)'); plt.title('Concreto — parábola-retângulo (8.2.10.1)')
plt.show()

## 3. Diagrama tensão-deformação do aço

**NBR 6118, 8.3.6** — diagrama bilinear (elastoplástico perfeito), igual na tração e na compressão:

$$\sigma_s = E_s\,\varepsilon_s \;\; \text{se } |\varepsilon_s| \le \varepsilon_{yd}; \qquad \sigma_s = \pm f_{yd} \;\; \text{se } |\varepsilon_s| > \varepsilon_{yd}$$

O alongamento máximo da armadura tracionada é **εsu = 10 ‰** (17.2.2, hipótese e — limite convencional do domínio 1/2).

In [ ]:
EPS_SU = 0.010

def tensao_aco(eps, fyd):
    return np.clip(ES_KGF_CM2 * eps, -fyd, fyd)

e = np.linspace(-0.012, 0.004, 300)
plt.figure(figsize=(6, 3.5))
plt.plot(e * 1000, tensao_aco(e, fyd), color=LARANJA, lw=2)
plt.axvline(-10, ls='--', color=CINZA, lw=1); plt.text(-10, -fyd * .9, ' εsu = 10 ‰', color=CINZA)
plt.xlabel('εs (‰)  — tração negativa'); plt.ylabel('σs (kgf/cm²)'); plt.title('Aço CA-50 — elastoplástico (8.3.6)')
plt.show()

## 4. Geometria: concreto em faixas e barras

A seção é dividida em **faixas horizontais** finas (240 no software). Cada faixa tem ordenada $y_i$ (a partir do centro) e área $A_i$. Para o círculo:

$$b(y) = 2\sqrt{R^2 - y^2} \qquad A_i = b(y_i)\,\Delta y$$

As barras ficam num círculo de raio

$$r_s = \frac{D}{2} - c - \phi_t - \frac{\phi}{2}$$

(cobrimento $c$, estribo $\phi_t$ e metade da barra longitudinal). Para n barras igualmente espaçadas: $y_j = r_s \sin(2\pi j/n)$.

Mínimo de **6 barras** em seção circular: NBR 6118, 18.4.2.2.

In [ ]:
def secao_circular(diametro_cm, n_faixas=240):
    # Faixas horizontais: ordenada do centro (cm) e área (cm²).
    r = diametro_cm / 2
    bordas = np.linspace(-r, r, n_faixas + 1)
    y = (bordas[:-1] + bordas[1:]) / 2
    area = 2 * np.sqrt(np.maximum(r * r - y * y, 0.0)) * np.diff(bordas)
    return y, area


def barras_circulares(n, raio_cm, giro=0.0):
    ang = giro + 2 * math.pi * np.arange(n) / n
    return raio_cm * np.sin(ang)


y_conc, a_conc = secao_circular(D)
raio_s = D / 2 - cobrimento - phi_estribo / 10 - phi / 20
y_bar = barras_circulares(n_barras, raio_s)
area_barra = math.pi * (phi / 10) ** 2 / 4
As = n_barras * area_barra
print(f'Área de concreto = {a_conc.sum():.1f} cm² (exata: {math.pi*D*D/4:.1f})')
print(f'Raio das barras = {raio_s:.2f} cm   As = {n_barras} × {area_barra:.3f} = {As:.2f} cm²   ρ = {As/(math.pi*D*D/4)*100:.2f} %')

ang = np.linspace(0, 2 * np.pi, 200)
xb = raio_s * np.cos(2 * np.pi * np.arange(n_barras) / n_barras)
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(D / 2 * np.cos(ang), D / 2 * np.sin(ang), color=CINZA)
for yy in y_conc[::12]:
    b = math.sqrt(max((D / 2) ** 2 - yy ** 2, 0)); ax.plot([-b, b], [yy, yy], color=AZUL, lw=.5, alpha=.5)
ax.plot(xb, y_bar, 'o', color=LARANJA, ms=7)
ax.set_aspect('equal'); ax.set_title('Faixas de concreto (1 a cada 12) e barras')
plt.show()

## 5. Estados-limite últimos: domínios de deformação (polos A, B e C)

**NBR 6118, 17.2.2** (hipóteses básicas e Figura 17.1): as seções permanecem planas (hipótese de Bernoulli), há aderência perfeita e a ruína ocorre quando:

- **Polo A** — o aço mais tracionado atinge εsu = 10 ‰ (domínios 1 e 2);
- **Polo B** — a fibra de concreto mais comprimida atinge εcu (domínios 3, 4 e 4a);
- **Polo C** — na compressão com a seção toda comprimida, a fibra situada a $\frac{\varepsilon_{cu}-\varepsilon_{c2}}{\varepsilon_{cu}}\,h$ da borda mais comprimida fica com εc2 (domínio 5).

Para percorrer **todas** as configurações de ruína em sequência, usa-se um parâmetro **t de 0 a 3**:

| t | Polo | O que varia | De → até |
|---|---|---|---|
| 0 → 1 | A (εs = −10 ‰ na barra mais baixa) | ε no topo | −10 ‰ (tração uniforme) → εcu |
| 1 → 2 | B (εc = εcu no topo) | profundidade da linha neutra x | x₂₃ → h |
| 2 → 3 | C (εc2 no ponto C) | ε na base | 0 → εc2 (compressão uniforme) |

Cada t define a deformação no topo $\varepsilon_{topo}$ e a curvatura $\kappa$; a deformação em qualquer ponto é

$$\varepsilon(y) = \varepsilon_{topo} - \kappa\,(y_{max} - y)$$

In [ ]:
def caminho_ultimo(t, h, d, ec2, ecu):
    # Deformação na fibra mais comprimida e curvatura ao longo do contorno
    # último: t 0→1 polo A (aço a 10 ‰), 1→2 polo B (εcu), 2→3 polo C (εc2).
    e_topo = np.empty_like(t)
    curv = np.empty_like(t)
    a = t <= 1                                   # polo A
    e_topo[a] = -EPS_SU + t[a] * (ecu + EPS_SU)
    curv[a] = (e_topo[a] + EPS_SU) / d
    b = (t > 1) & (t <= 2)                       # polo B
    xa = ecu / (ecu + EPS_SU) * d                # LN na fronteira dos domínios 2/3
    x = xa + (t[b] - 1) * (h - xa)
    e_topo[b] = ecu
    curv[b] = ecu / x
    c = t > 2                                    # polo C
    e_base = (t[c] - 2) * ec2
    hc = (1 - ec2 / ecu) * h
    curv[c] = (ec2 - e_base) / (h - hc)
    e_topo[c] = ec2 + curv[c] * hc
    return e_topo, curv


h = D
ymax = D / 2
d = ymax - y_bar.min()                           # altura útil (barra mais baixa)
print(f'h = {h:.1f} cm   d = {d:.2f} cm')

fig, ax = plt.subplots(figsize=(7, 4.2))
for tt, cor in [(0, LARANJA), (0.5, LARANJA), (1, CINZA), (1.5, AZUL), (2, AZUL), (2.5, VERDE), (3, VERDE)]:
    et, k = caminho_ultimo(np.array([tt], float), h, d, ec2, ecu)
    ys = np.array([-D / 2, D / 2])
    ax.plot((et[0] - k[0] * (ymax - ys)) * 1000, ys, color=cor, lw=1.6, label=f't = {tt}')
ax.axvline(0, color='k', lw=.8); ax.axhline(y_bar.min(), ls=':', color=LARANJA)
ax.text(-9.8, y_bar.min() + 1, 'barra mais tracionada', color=LARANJA, fontsize=8)
ax.set_xlabel('ε (‰) — compressão +'); ax.set_ylabel('y (cm)'); ax.legend(fontsize=8, ncol=2)
ax.set_title('Configurações de ruína (Fig. 17.1 da NBR 6118)')
plt.show()

## 6. Integração dos esforços resistentes

Para cada configuração de deformação (cada t), as tensões de cada faixa e de cada barra são obtidas pelos diagramas das etapas 2 e 3, e os esforços resistentes vêm do **equilíbrio** da seção:

$$N_R = \sum_i \sigma_c(\varepsilon_i)\,A_i + \sum_j \sigma_s(\varepsilon_j)\,A_{s,j}$$

$$M_R = \sum_i \sigma_c(\varepsilon_i)\,A_i\,y_i + \sum_j \sigma_s(\varepsilon_j)\,A_{s,j}\,y_j$$

(momento em relação ao centro da seção). Como todas as barras têm a mesma área, a parcela do aço é **linear em As**:

$$N_R(t) = N_c(t) + A_s\,n_s(t) \qquad M_R(t) = M_c(t) + A_s\,m_s(t)$$

Isso permite calcular a parte do concreto uma única vez e testar qualquer As rapidamente — é o que faz a classe `Envoltoria`.

> Simplificação usual: a área de concreto ocupada pelas barras comprimidas não é descontada (efeito desprezível para taxas baixas).

In [ ]:
class Envoltoria:
    # Pré-calcula, ao longo do contorno último (t de 0 a 3), as parcelas
    # do concreto (Nc, Mc) e as do aço por cm² de armadura (ns, ms).
    def __init__(self, y_conc, a_conc, y_barras, sigma_cd, fyd, fck_mpa, n_pontos=900):
        ec2, ecu, nexp = parametros_concreto(fck_mpa)
        passo = float(y_conc[1] - y_conc[0])
        ymax_c, ymin_c = float(np.max(y_conc)) + passo / 2, float(np.min(y_conc)) - passo / 2
        h = ymax_c - ymin_c
        d = ymax_c - float(np.min(y_barras))
        t = np.linspace(0.0, 3.0, n_pontos)
        e_topo, curv = caminho_ultimo(t, h, d, ec2, ecu)
        # deformações: linhas = configurações (t), colunas = faixas / barras
        eps_c = e_topo[:, None] - curv[:, None] * (ymax_c - y_conc[None, :])
        sig_c = tensao_concreto(eps_c, sigma_cd, ec2, nexp)
        self.Nc = sig_c @ a_conc
        self.Mc = sig_c @ (a_conc * y_conc)
        eps_s = e_topo[:, None] - curv[:, None] * (ymax_c - y_barras[None, :])
        sig_s = tensao_aco(eps_s, fyd)
        nb = len(y_barras)
        self.ns = sig_s.sum(axis=1) / nb          # por cm² de As total
        self.ms = (sig_s @ y_barras) / nb
        self.t = t

    def curva(self, As):
        return self.Nc + As * self.ns, self.Mc + As * self.ms

    def momento_resistente(self, Nd, As):
        # MRd (kgf·cm) para Nd (kgf); -inf se Nd fora da capacidade da seção.
        N, M = self.curva(As)
        if not (N[0] <= Nd <= N[-1]):
            return -math.inf
        i = min(max(int(np.searchsorted(N, Nd)), 1), len(N) - 1)
        f = 0.0 if N[i] == N[i - 1] else (Nd - N[i - 1]) / (N[i] - N[i - 1])
        return float(M[i - 1] + f * (M[i] - M[i - 1]))

    def as_necessaria(self, Nd, Md, as_max):
        # Menor As (cm²) com (Nd, |Md|) dentro da envoltória — bisseção.
        Md = abs(Md)
        if self.momento_resistente(Nd, 0.0) >= Md:
            return 0.0
        if self.momento_resistente(Nd, as_max) < Md:
            return None
        lo, hi = 0.0, as_max
        for _ in range(60):
            m = (lo + hi) / 2
            if self.momento_resistente(Nd, m) >= Md:
                hi = m
            else:
                lo = m
        return hi


env = Envoltoria(y_conc, a_conc, y_bar, sigma_cd, fyd, fck)

# Exemplo de uma configuração: t = 1,5 (domínio 3/4)
i = np.searchsorted(env.t, 1.5)
N, M = env.curva(As)
print(f't = 1,5 → N_R = {N[i]/1000:.1f} tf   M_R = {M[i]/1e5:.2f} tf·m')
print(f'Tração pura (t = 0): N_R = {N[0]/1000:.1f} tf  (= −As·fyd = {-As*fyd/1000:.1f} tf)')
sig_s_c2 = min(fyd, ES_KGF_CM2 * ec2)   # aço limitado a εc2 = 2 ‰ → 4.200 kgf/cm² < fyd
print(f'Compressão pura (t = 3): N_R = {N[-1]/1000:.1f} tf  (= 0,85fcd·Ac + As·σs(εc2) = {(sigma_cd*math.pi*D*D/4 + As*sig_s_c2)/1000:.1f} tf)')

## 7. A envoltória N × M

Percorrendo t de 0 a 3, os pares $(N_R, M_R)$ desenham a **envoltória resistente** (diagrama de interação). Pontos notáveis:

- **Tração pura** (t = 0): $N_R = -A_s f_{yd}$, M = 0;
- **Ponto de balanceamento**: maior momento resistente — concreto em εcu e aço tracionado começando a escoar (fronteira dos domínios 3/4);
- **Compressão pura** (t = 3): $N_R = \alpha_c f_{cd} A_c + A_s\,\sigma_s(\varepsilon_{c2})$ — com εc2 = 2 ‰ o CA-50 fica em $E_s\varepsilon_{c2}$ = 4.200 kgf/cm², abaixo de fyd.

**Verificação no ELU (17.2.5 / 17.2.2):** a seção atende se o ponto solicitante $(N_d, M_d)$ fica **dentro** da envoltória.

In [ ]:
N, M = env.curva(As)
ib = int(np.argmax(M))
casos = [('compressão 1', 22550, 14656), ('compressão 2', 22528, 15470),
         ('tração 1', -17787, 12829), ('tração 2T', -14982, 14369)]   # Nd (kgf), Md (kgf·m) — solo I

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.fill(np.r_[M / 1e5, 0], np.r_[N / 1000, N[0] / 1000], color=AZUL, alpha=.08)
ax.plot(M / 1e5, N / 1000, color=AZUL, lw=2, label=f'Envoltória — {n_barras} Ø {phi:g}')
ax.plot(M[ib] / 1e5, N[ib] / 1000, 's', color=AZUL); ax.annotate('balanceamento', (M[ib] / 1e5, N[ib] / 1000), xytext=(-95, 10), textcoords='offset points')
desloc = [(-78, 8), (8, 8), (-60, -14), (8, -14)]
for (nome, nd, mdk), dxy in zip(casos, desloc):
    ax.plot(mdk / 1000, nd / 1000, 'o', color=VERDE if nd > 0 else LARANJA, ms=7)
    ax.annotate(nome, (mdk / 1000, nd / 1000), xytext=dxy, textcoords='offset points', fontsize=8)
ax.axhline(0, color='k', lw=.8)
ax.set_xlabel('M (tf·m)'); ax.set_ylabel('N (tf) — compressão +'); ax.legend(loc='upper left')
ax.set_title('Envoltória resistente N × M (17.2.2)')
plt.show()
print(f'Balanceamento: N = {N[ib]/1000:.1f} tf, MRd,máx = {M[ib]/1e5:.2f} tf·m')

## 8. Momento resistente para um Nd e fator de utilização

Para uma hipótese com esforço normal $N_d$, o **MRd** é o ponto da envoltória na mesma altura $N = N_d$ (interpolação entre os dois pontos vizinhos da curva). A folga da armadura é medida por

$$\eta = \frac{M_d}{M_{Rd}(N_d)} \le 1$$

In [ ]:
print(f'{"Hipótese":<14}{"Nd (tf)":>9}{"Md (tf·m)":>11}{"MRd (tf·m)":>12}{"Md/MRd":>9}')
for nome, nd, mdk in casos:
    mrd = env.momento_resistente(nd, As) / 100          # kgf·m
    print(f'{nome:<14}{nd/1000:>9.1f}{mdk/1000:>11.2f}{mrd/1000:>12.2f}{mdk/mrd:>9.2f}')

## 9. Armadura necessária (dimensionamento)

O dimensionamento é o problema inverso: achar a **menor As** tal que $M_{Rd}(N_d, A_s) \ge M_d$. Como MRd cresce com As, usa-se **bisseção**.

No software, essa análise é feita uma vez com um anel denso de barras (48) no raio da maior bitola disponível (a favor da segurança). Depois:

- **As mínima** (17.3.5.3.1): o software usa a taxa mínima da planilha (0,4 % Ac);
- **As máxima** (17.3.5.3.2): 8 % Ac, incluindo a região de emendas;
- as barras saem de As,req = máx(As,calc; As,mín) pelo critério de espaçamento da planilha.

A relação com o ábaco é $\omega = \dfrac{A_s f_{yd}}{A_c\,\alpha_c f_{cd}}$.

In [ ]:
anel = Envoltoria(y_conc, a_conc, barras_circulares(48, D/2 - cobrimento - phi_estribo/10 - 25/20), sigma_cd, fyd, fck)
Ac = math.pi * D * D / 4
for nome, nd, mdk in casos:
    a = anel.as_necessaria(nd, mdk * 100, 0.08 * Ac)
    print(f'{nome:<14} As,nec = {a:6.2f} cm²   ω = {a*fyd/(Ac*sigma_cd):.3f}')
print(f'As,mín = 0,4 % Ac = {0.004*Ac:.2f} cm²   As,máx = 8 % Ac = {0.08*Ac:.1f} cm²')

## 10. Flexão oblíqua — envoltória Mx × My

Na **flexão composta oblíqua** (17.2.5.2) a linha neutra pode ter qualquer inclinação. Para um $N_d$ fixo:

1. gira-se a linha neutra de um ângulo α (0 a 360°);
2. para cada α, projetam-se as coordenadas na direção perpendicular à LN: $v = x\sin\alpha + y\cos\alpha$;
3. percorre-se o caminho último (etapa 5) nessa direção e acha-se a configuração com $N_R = N_d$;
4. calculam-se **os dois momentos** dessa configuração:

$$M_x = \sum \sigma\,A\,y \qquad M_y = \sum \sigma\,A\,x$$

Os pares (Mx, My) formam o **contorno resistente** no plano Mx × My. Aqui a seção é discretizada em **fibras** (quadradinhos), porque as faixas horizontais só servem para uma direção.

In [ ]:
def fibras_circulo(diametro_cm, n=44):
    r = diametro_cm / 2
    passo = diametro_cm / n
    g = -r + passo * (np.arange(n) + 0.5)
    x, y = np.meshgrid(g, g)
    dentro = x ** 2 + y ** 2 <= r * r
    x, y = x[dentro], y[dentro]
    a = np.full(x.shape, math.pi * r * r / len(x))
    ang = np.linspace(0, 2 * math.pi, 181)
    return (x, y, a), (r * np.cos(ang), r * np.sin(ang))


def contorno_mxmy(fibras, borda, barras, area_barra_cm2, lista_nd, sigma_cd, fyd, fck_mpa, n_ang=40, n_t=260):
    xc, yc, ac = fibras
    bx, by = borda
    xs, ys = barras
    ec2, ecu, nexp = parametros_concreto(fck_mpa)
    t = np.linspace(0.0, 3.0, n_t)
    saida = [[] for _ in lista_nd]
    for alfa in np.linspace(0, 2 * math.pi, n_ang, endpoint=False):
        s, c = math.sin(alfa), math.cos(alfa)
        v_c, v_b, v_s = xc * s + yc * c, bx * s + by * c, xs * s + ys * c     # passo 2
        vmax = float(v_b.max()); h = vmax - float(v_b.min()); d = vmax - float(v_s.min())
        e_topo, curv = caminho_ultimo(t, h, d, ec2, ecu)                      # passo 3
        sig_c = tensao_concreto(e_topo[:, None] - curv[:, None] * (vmax - v_c[None, :]), sigma_cd, ec2, nexp) * ac
        sig_s = tensao_aco(e_topo[:, None] - curv[:, None] * (vmax - v_s[None, :]), fyd) * area_barra_cm2
        N = sig_c.sum(axis=1) + sig_s.sum(axis=1)
        Mx = (sig_c @ yc + sig_s @ ys) / 100                                  # passo 4 (kgf·m)
        My = (sig_c @ xc + sig_s @ xs) / 100
        for k, nd in enumerate(lista_nd):
            if not (N[0] <= nd <= N[-1]):
                continue
            i = min(max(int(np.searchsorted(N, nd)), 1), len(N) - 1)
            f = 0.0 if N[i] == N[i - 1] else (nd - N[i - 1]) / (N[i] - N[i - 1])
            saida[k].append([Mx[i - 1] + f * (Mx[i] - Mx[i - 1]), My[i - 1] + f * (My[i] - My[i - 1])])
    return [np.array(p) for p in saida]


fib, borda = fibras_circulo(D)
ang_b = 2 * math.pi * np.arange(n_barras) / n_barras
barras_xy = (raio_s * np.cos(ang_b), raio_s * np.sin(ang_b))
cont_c, cont_t = contorno_mxmy(fib, borda, barras_xy, area_barra, [22528, -14982], sigma_cd, fyd, fck)
R = np.hypot(cont_c[:, 0], cont_c[:, 1])
print(f'Nd = 22,5 tf: MRd no contorno entre {R.min()/1000:.2f} e {R.max()/1000:.2f} tf·m '
      f'(faixas, etapa 8: {env.momento_resistente(22528, As)/1e5:.2f})')

## 11. Momento mínimo e envoltórias mínimas

**Momento mínimo de 1ª ordem — NBR 6118, 11.3.3.4.3** (imperfeições geométricas locais):

$$M_{1d,mín} = N_d\,(0{,}015 + 0{,}03\,h) \qquad (h \text{ em metros, na direção considerada})$$

Na flexão oblíqua, a **envoltória mínima de 1ª ordem** é a elipse

$$\left(\frac{M_{1d,mín,x}}{M_{1d,mín,xx}}\right)^2 + \left(\frac{M_{1d,mín,y}}{M_{1d,mín,yy}}\right)^2 = 1$$

que vira um **círculo** na seção circular (h = D nas duas direções).

**Efeito local de 2ª ordem — pilar-padrão com curvatura aproximada (15.8.3.3.2):**

$$\frac{1}{r} = \frac{0{,}005}{h\,(\nu + 0{,}5)} \le \frac{0{,}005}{h} \qquad \nu = \frac{N_d}{A_c f_{cd}} \qquad M_{d,tot} = \alpha_b M_{1d,A} + N_d\,\frac{\ell_e^2}{10}\,\frac{1}{r} \ge M_{1d,A}$$

**Dispensa (15.8.2):** não é preciso considerar a 2ª ordem se $\lambda \le \lambda_1$, com

$$\lambda = \frac{\ell_e}{i} \qquad \lambda_1 = \frac{25 + 12{,}5\,e_1/h}{\alpha_b}, \quad 35 \le \lambda_1 \le 90$$

- tubulão tratado como **balanço**: $\ell_e = 2\ell$ (ℓ = afloramento + profundidade do momento máximo) e $i = D/4$;
- balanço: $\alpha_b = 0{,}80 + 0{,}20\,M_C/M_A \ge 0{,}85$ → com carga horizontal no topo, $M_C = M_A/2$ e **αb = 0,90**.

**Envoltória mínima com 2ª ordem (15.3.2):** $M_{1d,mín} + M_{2d}$, calculada com αb = 1,0 e λ1 obtido com a excentricidade mínima. A envoltória resistente deve **englobar** essa envoltória.

> Hipótese de projeto (não é procedimento da norma para tubulões): o fuste foi tratado como balanço engastado na profundidade do momento máximo.

In [ ]:
# Hipótese 2 de compressão, solo I (dados do software)
Nd = 22528.0                        # kgf = 1,1 × 20.480
M1d = 15470.0                       # kgf·m — momento de 1ª ordem na seção crítica
H = math.hypot(3160, 1380)          # kgf — resultante horizontal (característica)
d_m = D / 100
m1d_min = Nd * (0.015 + 0.03 * d_m)                         # kgf·m
# comprimento do balanço: afloramento máx. + profundidade do momento máximo (planilha)
afl, gamma_solo, Kp = 3.5, 1700.0, 3.0                      # solo I: γ = 1700 kgf/m³, φ = 30° → Kp = 3
ell = afl + 0.545 * math.sqrt(H / (gamma_solo * d_m * Kp))
le = 2 * ell
lam = le / (d_m / 4)
lam1_min = min(90, max(35, 25 + 12.5 * (0.015 + 0.03 * d_m) / d_m))    # αb = 1,0
nu = Nd / (Ac * fck / gamma_c * 10)
curv = min(0.005 / (d_m * (nu + 0.5)), 0.005 / d_m)
m2d = Nd * le ** 2 / 10 * curv
m_min_2a = m1d_min + (m2d if lam > lam1_min else 0)
print(f'M1d,mín = {m1d_min:.0f} kgf·m   ℓe = {le:.2f} m   λ = {lam:.1f}   λ1 = {lam1_min:.1f}   ν = {nu:.3f}')
print(f'1/r = {curv:.5f} 1/m   M2d = {m2d:.0f} kgf·m   Envoltória mínima com 2ª ordem = {m_min_2a:.0f} kgf·m')
lam1 = min(90, max(35, (25 + 12.5 * (max(M1d, m1d_min) / Nd) / d_m) / 0.9))             # αb = 0,90
md_tot = max(M1d, 0.9 * M1d + m2d) if lam > lam1 else M1d
print(f'Hipótese 2: λ1 = {lam1:.1f} → 2ª ordem {"necessária" if lam > lam1 else "dispensada"}; '
      f'Md,tot = máx({M1d:.0f}; 0,9×{M1d:.0f} + {m2d:.0f}) = {md_tot:.0f} kgf·m')

tt = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(np.r_[cont_c[:, 0], cont_c[0, 0]] / 1000, np.r_[cont_c[:, 1], cont_c[0, 1]] / 1000, color=VERDE, lw=2, label='Resistente (Nd = 22,5 tf)')
ax.plot(np.r_[cont_t[:, 0], cont_t[0, 0]] / 1000, np.r_[cont_t[:, 1], cont_t[0, 1]] / 1000, color=LARANJA, lw=2, label='Resistente (Nd = −15 tf)')
ax.plot(m1d_min / 1000 * np.cos(tt), m1d_min / 1000 * np.sin(tt), '--', color=CINZA, label='Mínima de 1ª ordem')
ax.plot(m_min_2a / 1000 * np.cos(tt), m_min_2a / 1000 * np.sin(tt), ':', color=CINZA, lw=1.8, label='Mínima com 2ª ordem')
ax.set_aspect('equal'); ax.axhline(0, color='k', lw=.8); ax.axvline(0, color='k', lw=.8)
ax.set_xlabel('Mx (tf·m)'); ax.set_ylabel('My (tf·m)'); ax.legend(fontsize=8, loc='lower left')
ax.set_title('Flexão oblíqua e envoltórias mínimas (15.3.2)')
plt.show()

## 12. Resumo do roteiro

1. Resistências de cálculo — 12.3.3
2. Concreto parábola-retângulo, αc — 8.2.10.1 e 17.2.2
3. Aço elastoplástico, εsu = 10 ‰ — 8.3.6 e 17.2.2
4. Seção em faixas/fibras e posição das barras — (mín. 6 barras: 18.4.2.2)
5. Configurações de ruína pelos polos A, B e C — 17.2.2 (Fig. 17.1)
6. Integração de N e M por equilíbrio — 17.2.2
7. Envoltória N × M e verificação do ponto (Nd, Md)
8. MRd(Nd) e fator de utilização Md/MRd
9. As necessária por bisseção; As,mín e As,máx — 17.3.5.3
10. Flexão oblíqua: contorno Mx × My — 17.2.5.2
11. M1d,mín — 11.3.3.4.3; 2ª ordem — 15.8.2 e 15.8.3.3.2; envoltória mínima com 2ª ordem — 15.3.2

**Bibliografia complementar sugerida:** a própria NBR 6118:2023 e seus comentários técnicos (IBRACON); material didático de pilares de L. M. Pinheiro (EESC-USP) — confira a edição disponível.